In [6]:
import os
import io
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.syntax import Syntax
from rich import box

In [10]:
#Load API Key
load_dotenv(".env")
open_api_key= os.getenv("OPENAI_API_KEY")

console=Console()

#Load Dataset
df= pd.read_csv("dataset.csv")

console.print("\n[bold green]Dataset loaded successfully[/bold green]\n")
table=Table(title="Dataset Preview", box=box.DOUBLE_EDGE)

for col in df.columns:
    table.add_column(col)

for _, row in df.iterrows():
    table.add_row(*[str(i) for i in row])

console.print(table)

llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)

Dataset loaded successfully

                        Dataset Preview                        
╔════════════╤═════════╤══════════╤═══════════════╤═══════════╗
║ date       │ sku     │ location │ quantity_sold │ inventory ║
╟────────────┼─────────┼──────────┼───────────────┼───────────╢
║ 01/01/2024 │ SKU_101 │ Chennai  │ 52.0          │ 120       ║
║ 02/01/2024 │ SKU_101 │ Chennai  │ 49.0          │ 115       ║
║ 03/01/2024 │ SKU_101 │ Chennai  │ nan           │ 110       ║
║ 04/01/2024 │ SKU_101 │ Chennai  │ 55.0          │ 105       ║
║ 05/01/2024 │ SKU_101 │ Chennai  │ 60.0          │ 100       ║
║ 06/01/2024 │ SKU_101 │ Chennai  │ -5.0          │ 95        ║
║ 07/01/2024 │ SKU_101 │ Chennai  │ 58.0          │ 90        ║
║ 08/01/2024 │ SKU_101 │ Chennai  │ 62.0          │ 85        ║
║ 09/01/2024 │ SKU_101 │ Chennai  │ 59.0          │ 80        ║
║ 10/01/2024 │ SKU_101 │ Chennai  │ 57.0          │ 75        ║
║ 11/01/2024 │ SKU_101 │ Chennai  │ 300.0         │ 70        ║
║ 12/01/2024 │ SKU_101 │ Chennai  │ 56.0          │ 65        ║
║ 13/01/2024 │ SKU_101 │ Chennai  │ 54.0          │ 60        ║
║ 14/01/2024 │ SKU_101 │ Chennai  │ 2.0           │ 55        ║
║ 15/01/2024 │ SKU_101 │ Chennai  │ 53.0          │ 50        ║
╚════════════╧═════════╧══════════╧═══════════════╧═══════════╝

In [18]:
# Agent 1 - Guding Agent

tasks= [
    "Display the entire dataset",
    "Print number of rows, number of columns and column names",
    "Identify and print quantitative and qualitative columns",
    "Detect missing values, print number of missing values and handle them",
    "Detect outliers, print high and low outliers and handle them using IQR",
    "Find duplicates and print number of duplicate rows",
]  

# Agent 2 - Coding Agent
def coder_agent(tasks):
    
    prompt= f"""
    You are an expert Python data preprocessing engineer.

    Rules:
    1. Dataframe name is df
    2. Use pandas and numpy
    3. Do NOT reload dataset
    4. Numeric columns = df.select_dtypes(include=np.number)
    5. Categorical columns = df.select_dtypes(include="object")
    6. Return only executable Python code

    Task:
    {task}
    """
    
    response=llm.invoke(prompt)
    code= response.content
    
    
    code = code.replace("```python", "").replace("```", "").strip()
    console.print("\n[bold yellow]Generated Code[/bold yellow]\n")

    syntax = Syntax(code, "python", theme="monokai", line_numbers=False)
    console.print(syntax)

    return code

# Agent 3 - Executor Agent

def executor_agent(code):

    global df

    try:
        local_vars={
            "df": df,
            "np": np,
            "pd": pd,
        }

         # Capture output
        import io
        import sys

        buffer= io.StringIO()
        sys.stdout= buffer

        exec(code, globals(),  local_vars)

        sys.stdout= sys.__stdout__

        df=local_vars.get("df", df)

        output= buffer.getvalue()

        console.print(Panel(output if output else "Execution Completed", title="Execution Output", style="bold cyan on black"))

    except Exception as e:

        console.print(Panel(str(e), title= "Execution Error", style="bold red"))

#checker agent
def checker_agent(iteration):

    if iteration < 6:
        console.print("\n[bold cyan]Checker Agent --> Pending Preprocessing[/bold cyan]")
    else:
        console.print("\n[bold green]Checker Agent --> Preprocessed Completely[/bold green]")

for i, task in enumerate(tasks, start=1):

    console.print(f"[bold magenta]====================================\nIteration {i}\n====================================[/bold magenta]")
    console.print(f"[bold blue]Guiding Agent --> {task}[/bold blue]")
    code= coder_agent(task)
    executor_agent(code)
    checker_agent(i)

console.print("\n[bold green]Final Cleaned Dataset[/bold green]\n")

final_table= Table(title="Processed Dataset", box= box.DOUBLE_EDGE)

for col in df.columns:
    final_table.add_column(col)

for _, row in df.iterrows():
    final_table.add_row(*[str(i) for i in row])

console.print(final_table)

output_file= "Final Dataset.csv"
df.to_csv(output_file, index=False)

====================================
Iteration 1
====================================

Guiding Agent --> Display the entire dataset

Generated Code

print(df)                                                                                                          

╭─────────────────────────────────────────────── Execution Output ────────────────────────────────────────────────╮
│           date      sku location  quantity_sold  inventory                                                      │
│ 0   01/01/2024  SKU_101  Chennai           52.0        120                                                      │
│ 1   02/01/2024  SKU_101  Chennai           49.0        115                                                      │
│ 2   03/01/2024  SKU_101  Chennai            NaN        110                                                      │
│ 3   04/01/2024  SKU_101  Chennai           55.0        105                                                      │
│ 4   05/01/2024  SKU_101  Chennai           60.0        100                                                      │
│ 5   06/01/2024  SKU_101  Chennai           -5.0         95                                                      │
│ 6   07/01/2024  SKU_101  Chennai           58.0         90                                                      │
│ 7   08/01/2024  SKU_101  Chennai           62.0         85                                                      │
│ 8   09/01/2024  SKU_101  Chennai           59.0         80                                                      │
│ 9   10/01/2024  SKU_101  Chennai           57.0         75                                                      │
│ 10  11/01/2024  SKU_101  Chennai          300.0         70                                                      │
│ 11  12/01/2024  SKU_101  Chennai           56.0         65                                                      │
│ 12  13/01/2024  SKU_101  Chennai           54.0         60                                                      │
│ 13  14/01/2024  SKU_101  Chennai            2.0         55                                                      │
│ 14  15/01/2024  SKU_101  Chennai           53.0         50                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

duplicate_rows = df.duplicated().sum()                                                                             
print(f'Number of duplicate rows: {duplicate_rows}')                                                               

╭─────────────────────────────────────────────── Execution Output ────────────────────────────────────────────────╮
│ Number of duplicate rows: 0                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Checker Agent --> Preprocessed Completely

Final Cleaned Dataset

                         Processed Dataset                         
╔════════════╤═════════╤══════════╤═══════════════════╤═══════════╗
║ date       │ sku     │ location │ quantity_sold     │ inventory ║
╟────────────┼─────────┼──────────┼───────────────────┼───────────╢
║ 01/01/2024 │ SKU_101 │ Chennai  │ 52.0              │ 120.0     ║
║ 02/01/2024 │ SKU_101 │ Chennai  │ 49.0              │ 115.0     ║
║ 03/01/2024 │ SKU_101 │ Chennai  │ 65.14285714285714 │ 110.0     ║
║ 04/01/2024 │ SKU_101 │ Chennai  │ 55.0              │ 105.0     ║
║ 05/01/2024 │ SKU_101 │ Chennai  │ 60.0              │ 100.0     ║
║ 06/01/2024 │ SKU_101 │ Chennai  │ 56.0              │ 95.0      ║
║ 07/01/2024 │ SKU_101 │ Chennai  │ 58.0              │ 90.0      ║
║ 08/01/2024 │ SKU_101 │ Chennai  │ 62.0              │ 85.0      ║
║ 09/01/2024 │ SKU_101 │ Chennai  │ 59.0              │ 80.0      ║
║ 10/01/2024 │ SKU_101 │ Chennai  │ 57.0              │ 75.0      ║
║ 11/01/2024 │ SKU_101 │ Chennai  │ 56.0              │ 70.0      ║
║ 12/01/2024 │ SKU_101 │ Chennai  │ 56.0              │ 65.0      ║
║ 13/01/2024 │ SKU_101 │ Chennai  │ 54.0              │ 60.0      ║
║ 14/01/2024 │ SKU_101 │ Chennai  │ 56.0              │ 55.0      ║
║ 15/01/2024 │ SKU_101 │ Chennai  │ 53.0              │ 50.0      ║
╚════════════╧═════════╧══════════╧═══════════════════╧═══════════╝